In [15]:
import numpy as np
import pandas as pd
import polars as pl

pd.set_option('display.max_columns', None)


### Read Data used for TLGRF

In [16]:
augmented_df_pd = pd.read_csv("../../data/augmented_us-counties-states_latest_variants.csv")

In [17]:
augmented_df_pd.head()

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_NOHSDP,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_MOBILE,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_FIPS_Code,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Closed_day_cares,Reopen_day_cares,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,Stay_at_home_order'_issued_but_did_not_specifically_restrict_movement_of_the_general_public,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Closed_businesses_overnight,Began_to_reopen_businesses,Religious_gatherings_exempt_without_clear_social_distance_mandate*,Face_mask_mandate_in_public_spaces,Face_mask_mandate_x2,Face_mask_mandate_enforced_by_fines,Face_mask_mandate_enforced_by_criminal_charge/citation,No_legal_enforcement_of_face_mask_mandate,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Ended_face_mask_mandate_x2,Attempted_to_prevent_local_governments_from_implementing_face_mask_orders,Banned_local_mask_mandates,Liquor_stores_remained_open,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Firearm_sellers_remained_open,Cannabis_dispensaries_considered_essential_business,Closed_restaurants_except_take_out,Reopen_restaurants,Initially_reopen_restaurants_for_outdoor_dining_only,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Allowed_businesses_to_reopen_overnight,Began_to_reclose_bars,Closed_bars_(x2),Closed_movie_theaters_(x2),Closed_hair_salons/barber_shops_(x2),Closed_gyms_(x2),Closed_restaurants_(x2),Reopened_restaurants_(x2),Reopened_bars_(x2),Reopened_gyms_(x2),Reopened_hair_salons/barber_shops_(x2),Reopened_movie_theaters_(x2),Closed_bars_(x3),Closed_restaurants_(x3),Reopened_bars_(x3),Reopened_restaurants_(x3),Mandate_quarantine_for_those_entering_the_state_from_specific_settings,Mandate_quarantine_for_all_individuals_entering_the_state,Date_all_mandated_quarantines_ended,Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_People_who_are_Incarcerated,Vaccine_allocation_phase:_Residents_of_Homeless_Shelters,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers

In [18]:
print(augmented_df_pd.columns)

Index(['fips', 'date', 'county', 'state', 'cases', 'deaths', 'datetime',
       'days_from_start', 'rolled_cases', 'LAT',
       ...
       'Variant % P.1', 'Variant % P.2', 'Variant % R.1', 'Variant % XBB',
       'Variant % XBB.1.16', 'Variant % XBB.1.5', 'Variant % XBB.1.5.1',
       'Variant % XBB.1.9.1', 'Variant % XBB.1.9.2', 'Variant % XBB.2.3'],
      dtype='object', length=478)


### CUSP Impute

In [19]:
def replace_nan_forward_fill(df, columns):
    # Replace NaN values with 0 until the first non-NaN value,
    # within each 'fips' group and by 'date' for the specified columns
    copy_df = df.copy()
    def fill_nan_forward_fill(group):
        # Forward fill NaN values within each group after replacing preceding NaNs with 0
        group_filled = group.fillna(0)
        group_filled = group_filled.mask(group_filled.eq(0)).fillna(method='ffill')
        return group_filled
    
    # Group by 'fips' and apply forward fill within each group
    filled_values = copy_df.groupby(['fips'])[columns].fillna(method="ffill").fillna(0)
    
    # Update the specified columns with filled values
    copy_df[columns] = filled_values
    
    return copy_df


In [20]:
cusp_start = augmented_df_pd.columns.get_loc("death")
cusp_end = augmented_df_pd.columns.get_loc("metrics.vaccinationsCompletedRatio")
print((cusp_start,cusp_end))

vaccination_start = augmented_df_pd.columns.get_loc("Date_all_mandated_quarantines_ended")
vaccination_end = augmented_df_pd.columns.get_loc("Expanded_scope_of_practice_of_certain_health_providers_to_administer_vaccines")
print((vaccination_start,vaccination_end))



(378, 419)
(168, 212)


In [21]:
impute_columns = augmented_df_pd.columns[list((vaccination_start,vaccination_end)) + list((cusp_start,cusp_end))]

In [22]:
imputed_augmented_df_pd = augmented_df_pd.groupby("fips").ffill().fillna(0)
imputed_augmented_df_pd["fips"] = augmented_df_pd["fips"]


In [23]:
imputed_augmented_df_pd

,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,LON,AREA_SQMI,E_TOTPOP,E_HU,E_HH,E_POV,E_UNEMP,E_PCI,E_NOHSDP,E_AGE65,E_AGE17,E_DISABL,E_SNGPNT,E_MINRTY,E_LIMENG,E_MUNIT,E_MOBILE,E_CROWD,E_NOVEH,E_GROUPQ,EP_POV,MP_POV,EP_UNEMP,MP_UNEMP,EP_PCI,MP_PCI,EP_NOHSDP,MP_NOHSDP,EP_AGE65,MP_AGE65,EP_AGE17,MP_AGE17,EP_DISABL,MP_DISABL,EP_SNGPNT,MP_SNGPNT,EP_MINRTY,MP_MINRTY,EP_LIMENG,MP_LIMENG,EP_MUNIT,MP_MUNIT,EP_MOBILE,MP_MOBILE,EP_CROWD,MP_CROWD,EP_NOVEH,MP_NOVEH,EP_GROUPQ,MP_GROUPQ,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,SPL_THEME1,RPL_THEME1,EPL_AGE65,EPL_AGE17,EPL_DISABL,EPL_SNGPNT,SPL_THEME2,RPL_THEME2,EPL_MINRTY,EPL_LIMENG,SPL_THEME3,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,SPL_THEME4,RPL_THEME4,SPL_THEMES,RPL_THEMES,F_POV,F_UNEMP,F_PCI,F_NOHSDP,F_THEME1,F_AGE65,F_AGE17,F_DISABL,F_SNGPNT,F_THEME2,F_MINRTY,F_LIMENG,F_THEME3,F_MUNIT,F_MOBILE,F_CROWD,F_NOVEH,F_GROUPQ,F_THEME4,F_TOTAL,E_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,State_FIPS_Code,State_of_emergency_issued,State_of_emergency_lifted,Date_closed_K-12_public_schools,Closed_day_cares,Reopen_day_cares,Date_banned_visitors_to_nursing_homes,Stay_at_home/_shelter_in_place,Stay_at_home_order'_issued_but_did_not_specifically_restrict_movement_of_the_general_public,End/relax_stay_at_home/shelter_in_place,Closed_other_non-essential_businesses,Closed_businesses_overnight,Began_to_reopen_businesses,Religious_gatherings_exempt_without_clear_social_distance_mandate*,Face_mask_mandate_in_public_spaces,Face_mask_mandate_x2,Face_mask_mandate_enforced_by_fines,Face_mask_mandate_enforced_by_criminal_charge/citation,No_legal_enforcement_of_face_mask_mandate,Face_mask_mandate_for_employees_of_public-facing_businesses,Ended_face_mask_mandate,Ended_face_mask_mandate_x2,Attempted_to_prevent_local_governments_from_implementing_face_mask_orders,Banned_local_mask_mandates,Liquor_stores_remained_open,Allowed_restaurants_to_sell_takeout_alcohol,Allowed_restaurants_to_deliver_alcohol,Firearm_sellers_remained_open,Cannabis_dispensaries_considered_essential_business,Closed_restaurants_except_take_out,Reopen_restaurants,Initially_reopen_restaurants_for_outdoor_dining_only,Closed_gyms,Reopened_gyms,Closed_movie_theaters,Reopened_movie_theaters,Closed_Bars,Reopen_bars,Reopened_hair_salons/barber_shops,Reopened_religious_gatherings,Reopened_other_non-essential_retail,Allowed_businesses_to_reopen_overnight,Began_to_reclose_bars,Closed_bars_(x2),Closed_movie_theaters_(x2),Closed_hair_salons/barber_shops_(x2),Closed_gyms_(x2),Closed_restaurants_(x2),Reopened_restaurants_(x2),Reopened_bars_(x2),Reopened_gyms_(x2),Reopened_hair_salons/barber_shops_(x2),Reopened_movie_theaters_(x2),Closed_bars_(x3),Closed_restaurants_(x3),Reopened_bars_(x3),Reopened_restaurants_(x3),Mandate_quarantine_for_those_entering_the_state_from_specific_settings,Mandate_quarantine_for_all_individuals_entering_the_state,Date_all_mandated_quarantines_ended,Date_vaccine_allocation_plan_last_updated,Vaccine_allocation_phase:_Frontline_Healthcare_Workers,Vaccine_allocation_phase:_Healthcare_Service_Workers,Vaccine_allocation_phase:_Home_Healthcare_Workers,Vaccine_allocation_phase:_Additional_Healthcare_Workers,Vaccine_allocation_phase:_Long-term_Care_Residents,Vaccine_allocation_phase:_EMS_Providers,Vaccine_allocation_phase:_Firefighters,Vaccine_allocation_phase:_Law_Enforcement_&_Public_Safety_Personnel,Vaccine_allocation_phase:_Correctional_Staff,Vaccine_allocation_phase:_People_who_are_Incarcerated,Vaccine_allocation_phase:_Residents_of_Homeless_Shelters,Vaccine_allocation_phase:_Adults_Ages_75+,Vaccine_allocation_phase:_Adults_Ages_65+,Vaccine_allocation_phase:_Adults_w/_High-Risk_Medical_Conditions,Vaccine_allocation_phase:_Pre-K-12_School_Employees,Vaccine_allocation_phase:_Higher_Education_Employees,Vaccine_allocation_phase:_Public_Transit_Workers,Vaccine_allocation_phase:_Food_Supply_Workers,Vaccine_allocation_phase:_Grocery_Store_Workers,Vaccine_allocation_phase:_Frontline_Essential_Workers,Vacc

In [24]:
imputed_augmented_df_pd["Vaccine_allocation_phase:_Residents_of_Homeless_Shelters"].unique()

array([0., 2., 3., 1., 4.])

In [25]:
augmented_df_pd.columns.get_loc("Vaccine_allocation_phase:_Residents_of_Homeless_Shelters")

180

In [26]:
imputed_augmented_df_pd["E_POV"].unique()

array([  8422.,  21653.,   6597., ...,    984.,   1171., 295165.])

In [27]:
imputed_augmented_df_pd.to_parquet("imputed_augmented_us-counties-states_latest_variants.parquet")

In [28]:
imputed_augmented_df_pd.to_csv("imputed_augmented_us-counties-states_latest_variants.csv")